### Build a basic chatbot with Langgraph(GRAPH API)

In [1]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START,END 
from langgraph.graph.message import add_messages

ImportError: module ''langchain_core._api'.'deprecation'' not found (No module named 'annotated_types')

In [ ]:
class State(TypedDict):
    # Messages have the type "list". The 'add_message' function
    # in the annotation defines how this state keys should be updated
    # (in this case, it appends messages to the list, rather than overwriting them)
    messages:Annotated[list,add_messages]

graph_builder=StateGraph(State)

In [ ]:
graph_builder

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model

llm=ChatGroq(model="llama3-8b-8192")

ModuleNotFoundError: No module named 'langchain_groq'

In [ ]:
llm

c:\Users\Durga\OneDrive\Desktop\Practice\Project\Langchain Practice\AgenticAIWorkspace\AgenticLanggraph\.venv\Scripts\python.exe


In [ ]:
llm = init_chat_model("groq:llama3-8b-8192")
llm

In [ ]:
#Node functionality
def chatbot(state:State):
    return {"messages":[llm.invoke(state["messages"])]}

In [ ]:
graph_builder=StateGraph(State)

#Adding node
graph_builder.add_node("llmchatbot",chatbot)

#Adding edges
graph_builder.add_edges(START,"llmchatbot")
graph_builder.add_edges("llmchatbot", END)

# compile the graph
graph=graph_builder.compile()

In [ ]:
# Visualsize the graph
from IPython.display import Image,display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    pass

In [ ]:
response=graph.invoke({"messages":"Hi"})

In [ ]:
response["messages"][-1].content

In [ ]:
for event in graph.stream({"messages":"Hi How are you?"})
    for value in event.values():
        print(value["messages"][-1].content)

### Chatbot with Tool

In [ ]:
from langchain_tavily import TavilySearch

tool=TavilySearch(max_results=2)
tool.invoke("What is langgraph?")

In [ ]:
## Custom function
def multiply(a:int, b:int)->int:
    """Multiply a and b
    Args:
        a(int): first int
        b(int): second int

    Returns:
        int: output int
    """
    return a*b

In [ ]:
tools=[tool,multiply]

In [ ]:
llm_with_tool = llm.bind_tools(tools)

In [ ]:
llm_with_tool

In [ ]:
# Stategraph
from langgraph.graph import StateGraph,START,END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

# Node definition
def tool_calling_llm(state:State):
    return {"messages":[llm_with_tool.invoke(state["messages"])]}

# Graph
builder = StateGraph(State)
builder.add_node("tool_calling_llm",)
builder.add_node("tools",ToolNode(tools))

# Add edges
builder.add_edges(START, "tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    # If the latest message (result) from assistant is a tool call -> tools condition routes to tools
    # If the latest message (result) from assistant is not a tool call -> tools_condition routes to END
    tools_condition
)
builder.add_edge("tools",END)

# Compile the graph
graph = builder.compile()

from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
response = graph.invoke({"messages":"What is the recent AI news?"})

In [ ]:
response['messages'][-1].content

In [ ]:
for m in response['messages']:
    m.pretty_print()


In [ ]:
response = graph.invoke({"messages":"What is 2 multiply by 3?"})

In [ ]:
for m in response['messages']:
    m.pretty_print()


In [ ]:
response = graph.invoke({"messages":"What is 2 multiply by 3 and then multiply by 10?"})
for m in response['messages']:
    m.pretty_print()


### ReAct Agent Architecture

In [ ]:
# Stategraph
from langgraph.graph import StateGraph,START,END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

# Node definition
def tool_calling_llm(state:State):
    return {"messages":[llm_with_tool.invoke(state["messages"])]}

# Graph
builder = StateGraph(State)
builder.add_node("tool_calling_llm",)
builder.add_node("tools",ToolNode(tools))

# Add edges
builder.add_edges(START, "tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    # If the latest message (result) from assistant is a tool call -> tools condition routes to tools
    # If the latest message (result) from assistant is not a tool call -> tools_condition routes to END
    tools_condition
)
builder.add_edge("tools","tool_calling_llm")

# Compile the graph
graph = builder.compile()

from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))

### Adding memory in Agentic Graph

In [ ]:
response=graph.invoke({"messages":"Hello my name is KRish"})
for m in response['messages']:
    m.pretty_print()

In [ ]:
response=graph.invoke({"messages":"Hello, what's my name?"})
for m in response['messages']:
    m.pretty_print()

#it does not remember the name 

In [ ]:
# Stategraph
from langgraph.graph import StateGraph,START,END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

# Node definition
def tool_calling_llm(state:State):
    return {"messages":[llm_with_tool.invoke(state["messages"])]}

# Graph
builder = StateGraph(State)
builder.add_node("tool_calling_llm",)
builder.add_node("tools",ToolNode(tools))

# Add edges
builder.add_edges(START, "tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    # If the latest message (result) from assistant is a tool call -> tools condition routes to tools
    # If the latest message (result) from assistant is not a tool call -> tools_condition routes to END
    tools_condition
)
builder.add_edge("tools","tool_calling_llm")

# Compile the graph
graph = builder.compile(checkpointer=memory)

from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
config={"configurable":{"thread_id":"1"}}
response = graph.invoke({"messages":"Hi my name is Krish"},config=config)
response

In [ ]:
response['messages'][-1].content

In [ ]:
response=graph.invoke({"messages":"Hey what is my name?"},config=config)
print(response['messages'][-1].content)

### Streaming

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
memory=MemorySaver()

In [ ]:
def superbot(State:State):
    return {"messages":{llm_with_tool.invoke(state['messages'])}}

In [ ]:
graph=StateGraph(State)

## node
graph.add_node("SuperBot",superbot)

## edges
graph.add_edge(START, "SuperBot")
graph.add_edge("SuperBot",END)

graph_builder=graph.compile(checkpointer=memory)

## display
from IPython.display import Image, display
display(Image(graph_builder.get_graph().draw_mermaid_png()))

### Streaming

Methods: stream() and astream()
- These methods are sync and async methods for streaming back results.

Additional parameters in streaming modes for graph state

- values: This streams the full state of the graph after each node is called

- updates: This stream updates to the state of the graoh after each node is called

In [ ]:
# Create a thread
config = {"configurable":{"thread_id":"3"}}

for chunk in graph_builder.stream({'messages':"Hi,my name is Durga and i Like cricket"},config.stream_mode="updates"):
    print(chunk)

In [ ]:
for chunk in graph_builder.stream({'messages':"Hi,my name is Durga and i Like cricket"},config.stream_mode="values"):
    print(chunk)

In [ ]:
# Create a thread
config = {"configurable":{"thread_id":"4"}}

for chunk in graph_builder.stream({'messages':"Hi,my name is Durga and i Like cricket"},config.stream_mode="updates"):
    print(chunk)